In [2]:
import pandas as pd
from sqlalchemy import create_engine


con = create_engine("mysql+pymysql://root:@localhost/import_practice")



C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


**Note: Try to avoid *GROUP BY* clause to solve the problems**

For the problems use the *Health Insurance Claim* dataset. You can get the details as well as the dataset from [here](https://www.kaggle.com/datasets/thedevastator/insurance-claim-analysis-demographic-and-health).

### **Problem 1:** What are the top 5 patients who claimed the highest insurance amounts?

### **Problem 2:** What is the average insurance claimed by patients based on the number of children they have?

### **Problem 3:** What is the highest and lowest claimed amount by patients in each region?

### **Problem 4:** What is the percentage of smokers in each age group?

### **Problem 5:** What is the difference between the claimed amount of each patient and the first claimed amount of that patient?

### **Problem 6:** For each patient, calculate the difference between their claimed amount and the average claimed amount of patients with the same number of children.

### **Problem 7:** Show the patient with the highest BMI in each region and their respective rank.

### **Problem 8:** Calculate the difference between the claimed amount of each patient and the claimed amount of the patient who has the highest BMI in their region.


In [5]:
pd.read_sql_query("SELECT * FROM (SELECT *, RANK() OVER(ORDER BY claim DESC) AS 'rank' FROM insurance) AS E WHERE E.rank<6", con=con)

,level_0,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,rank
0,1339,1339,1340,30.0,female,47.4,101,No,0,Yes,southeast,63770.43,1
1,1338,1338,1339,37.0,male,30.4,106,No,0,Yes,southeast,62592.87,2
2,1337,1337,1338,30.0,male,34.5,91,Yes,3,Yes,northwest,60021.40,3
3,1336,1336,1337,59.0,female,38.1,120,No,1,Yes,northeast,58571.07,4
4,1335,1335,1336,44.0,female,35.5,88,Yes,0,Yes,northwest,55135.40,5


In [7]:
pd.read_sql_query("SELECT *, AVG(claim) OVER(PARTITION BY children) FROM insurance", con=con)

,level_0,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,AVG(claim) OVER(PARTITION BY children)
0,0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87,12327.993160
1,1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51,12327.993160
2,2,2,3,NaN,male,33.3,82,Yes,0,No,southeast,1135.94,12327.993160
3,3,3,4,NaN,male,33.7,80,No,0,No,northwest,1136.40,12327.993160
4,4,4,5,NaN,male,34.1,100,No,0,No,northwest,1137.01,12327.993160
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1335,718,718,719,34.0,male,25.8,100,Yes,5,No,southwest,10096.97,8786.035556
1336,815,815,816,29.0,female,31.9,90,Yes,5,No,southwest,11552.90,8786.035556
1337,880,880,881,28.0,female,46.8,106,No,5,No,southeast,12592.53,8786.035556
1338,975,975,976,50.0,male,25.5,84,No,5,No,southeast,14478.33,8786.035556


In [8]:
pd.read_sql_query("SELECT *, MAX(claim) OVER(PARTITION BY region) AS 'reg_max', MIN(claim) OVER(PARTITION BY region) AS 'reg_min' FROM insurance;", con=con)

,level_0,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,reg_max,reg_min
0,13,13,14,32.0,male,27.6,100,No,0,No,NaN,1252.41,1256.30,1252.41
1,14,14,15,40.0,male,28.7,81,Yes,0,No,NaN,1253.94,1256.30,1252.41
2,15,15,16,32.0,male,30.4,86,Yes,0,No,NaN,1256.30,1256.30,1252.41
3,48,48,49,34.0,male,16.0,83,No,0,No,northeast,1694.80,58571.07,1694.80
4,49,49,50,30.0,male,21.5,83,Yes,0,No,northeast,1702.46,58571.07,1694.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1335,1323,1323,1324,49.0,female,33.8,107,No,1,Yes,southwest,47928.03,52590.83,1261.44
1336,1324,1324,1325,39.0,male,39.9,115,No,0,Yes,southwest,48173.36,52590.83,1261.44
1337,1329,1329,1330,52.0,female,37.7,109,Yes,0,Yes,southwest,48824.45,52590.83,1261.44
1338,1333,1333,1334,44.0,male,36.4,127,No,1,Yes,southwest,51194.56,52590.83,1261.44


In [9]:
pd.read_sql_query("SELECT *, (COUNT(*) OVER(PARTITION BY age)/COUNT(*) OVER())*100 FROM insurance WHERE smoker='Yes';", con=con)

,level_0,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,(COUNT(*) OVER(PARTITION BY age)/COUNT(*) OVER())*100
0,1331,1331,1332,18.0,male,41.1,104,No,1,Yes,southeast,48970.25,0.3650
1,1016,1016,1017,19.0,male,23.0,99,Yes,2,Yes,northwest,17361.77,2.5547
2,1021,1021,1022,19.0,male,24.0,80,No,3,Yes,southeast,17663.14,2.5547
3,1045,1045,1046,19.0,male,25.3,95,No,2,Yes,southeast,18972.50,2.5547
4,1068,1068,1069,19.0,male,26.4,87,Yes,0,Yes,northeast,20149.32,2.5547
...,...,...,...,...,...,...,...,...,...,...,...,...,...
269,1008,1008,1009,60.0,female,27.9,97,No,0,Yes,southwest,16884.92,1.8248
270,1046,1046,1047,60.0,female,18.3,82,Yes,5,Yes,southwest,19023.26,1.8248
271,1104,1104,1105,60.0,female,28.1,84,No,1,Yes,northeast,22331.57,1.8248
272,1224,1224,1225,60.0,female,32.5,97,No,0,Yes,northwest,36898.73,1.8248


In [10]:
pd.read_sql_query("SELECT *, claim - FIRST_VALUE(claim) OVER() FROM insurance", con=con)

,level_0,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,claim - FIRST_VALUE(claim) OVER()
0,0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87,0.00
1,1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51,9.64
2,2,2,3,NaN,male,33.3,82,Yes,0,No,southeast,1135.94,14.07
3,3,3,4,NaN,male,33.7,80,No,0,No,northwest,1136.40,14.53
4,4,4,5,NaN,male,34.1,100,No,0,No,northwest,1137.01,15.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1335,1335,1335,1336,44.0,female,35.5,88,Yes,0,Yes,northwest,55135.40,54013.53
1336,1336,1336,1337,59.0,female,38.1,120,No,1,Yes,northeast,58571.07,57449.20
1337,1337,1337,1338,30.0,male,34.5,91,Yes,3,Yes,northwest,60021.40,58899.53
1338,1338,1338,1339,37.0,male,30.4,106,No,0,Yes,southeast,62592.87,61471.00


In [11]:
pd.read_sql_query("SELECT *, claim - AVG(claim) OVER(PARTITION BY children) FROM insurance;", con=con)

,level_0,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,claim - AVG(claim) OVER(PARTITION BY children)
0,0,0,1,39.0,male,23.2,91,Yes,0,No,southeast,1121.87,-11206.123160
1,1,1,2,24.0,male,30.1,87,No,0,No,southeast,1131.51,-11196.483160
2,2,2,3,NaN,male,33.3,82,Yes,0,No,southeast,1135.94,-11192.053160
3,3,3,4,NaN,male,33.7,80,No,0,No,northwest,1136.40,-11191.593160
4,4,4,5,NaN,male,34.1,100,No,0,No,northwest,1137.01,-11190.983160
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1335,718,718,719,34.0,male,25.8,100,Yes,5,No,southwest,10096.97,1310.934444
1336,815,815,816,29.0,female,31.9,90,Yes,5,No,southwest,11552.90,2766.864444
1337,880,880,881,28.0,female,46.8,106,No,5,No,southeast,12592.53,3806.494444
1338,975,975,976,50.0,male,25.5,84,No,5,No,southeast,14478.33,5692.294444


In [12]:
pd.read_sql_query("SELECT region, MAX(bmi), ranks FROM (SELECT *, RANK() OVER(ORDER BY bmi DESC) AS 'ranks' FROM insurance) AS t GROUP BY region", con=con)

,region,MAX(bmi),ranks
0,NaN,30.4,667
1,northeast,48.1,5
2,northwest,53.1,1
3,southeast,52.6,2
4,southwest,47.6,7


In [13]:
pd.read_sql_query("SELECT *, claim - FIRST_VALUE(claim) OVER(PARTITION BY region ORDER BY bmi DESC) FROM insurance", con=con)

,level_0,index,PatientID,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,claim - FIRST_VALUE(claim) OVER(PARTITION BY region ORDER BY bmi DESC)
0,15,15,16,32.0,male,30.4,86,Yes,0,No,NaN,1256.30,0.00
1,14,14,15,40.0,male,28.7,81,Yes,0,No,NaN,1253.94,-2.36
2,13,13,14,32.0,male,27.6,100,No,0,No,NaN,1252.41,-3.89
3,675,675,676,49.0,female,48.1,81,Yes,2,No,northeast,9432.93,0.00
4,681,681,682,27.0,female,44.7,98,No,0,No,northeast,9541.70,108.77
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1335,62,62,63,51.0,female,18.6,98,Yes,0,No,southwest,1728.90,-44384.61
1336,342,342,343,33.0,female,18.5,94,Yes,1,No,southwest,4766.02,-41347.49
1337,1046,1046,1047,60.0,female,18.3,82,Yes,5,Yes,southwest,19023.26,-27090.25
1338,61,61,62,32.0,female,17.8,90,No,0,No,southwest,1727.79,-44385.72
